In [49]:
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr
from shapely.ops import unary_union
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

In [67]:
# CMAP county shapefile
cmap_cty = gpd.read_file('C:/Users/x12la/Desktop/Scripts/CMAP_cty.shp')
cmap_cty = cmap_cty.to_crs("EPSG:4326")
cook_cty = cmap_cty.loc[cmap_cty["NAME"] == "Cook"].copy()
# One polygon representing the full CMAP domain

In [ ]:
primary_roads = gpd.read_file('C:/Users/x12la/Desktop/Scripts/tl_2016_us_primaryroads.shp')
primary_roads = primary_roads.to_crs('EPSG:4326')

In [68]:
cmap_roads = gpd.clip(primary_roads.to_crs("EPSG:4326"), cmap_cty)
cook_roads = gpd.clip(primary_roads.to_crs("EPSG:4326"), cook_cty)

cmap_union = cmap_cty.geometry.unary_union

In [74]:
model_dir = Path("./")
aqs_dir = Path("./")
output_dir = Path("./near_road_validation_output")

grid_file = model_dir / "latlon_ChicagoLADCO_d03.nc"

model_files = {
    "202301": {
        "BASE": model_dir / "all_202301_base_reduced.nc",
        "LOCUS": model_dir / "all_202301_locus_reduced.nc",},
    "202304": {
        "BASE": model_dir / "all_202304_base_reduced.nc",
        "LOCUS": model_dir / "all_202304_locus_reduced.nc",},
    "202307": {
        "BASE": model_dir / "all_202307_base_reduced.nc",
        "LOCUS": model_dir / "all_202307_locus_reduced.nc",},
    "202310": {
        "BASE": model_dir / "all_202310_base_reduced.nc",
        "LOCUS": model_dir / "all_202310_locus_reduced.nc",},}

dates = {
    "202301": ("2023-01-01 00:00", "2023-01-31 23:00"),
    "202304": ("2023-04-01 00:00", "2023-04-30 23:00"),
    "202307": ("2023-07-01 00:00", "2023-07-31 23:00"),
    "202310": ("2023-10-01 00:00", "2023-10-31 23:00"),}

aqs_files = {
    "NO2": aqs_dir / "hourly_42602_2023.csv",
    "PM25_TOT": aqs_dir / "hourly_88101_2023.csv",}

output_dir.mkdir(exist_ok=True)

In [75]:
grid = xr.open_dataset(grid_file)
lon_grid = grid["lon"].values
lat_grid = grid["lat"].values

grid.close()

In [76]:
# From Stacys original code 
def find_index(station_lon, station_lat, model_lon, model_lat):
    absolute_latitude_difference = np.abs(model_lat - station_lat)
    absolute_longitude_difference = np.abs(model_lon - station_lon)
    combined_difference = np.maximum(absolute_latitude_difference, absolute_longitude_difference,)
    row, col = np.where(combined_difference == combined_difference.min())
    return row[0], col[0]

In [77]:
def make_site_id(df):
    state = (df["State Code"].astype(int).astype(str).str.zfill(2))
    county = (df["County Code"].astype(int).astype(str).str.zfill(3))
    site = (df["Site Num"].astype(int).astype(str).str.zfill(4))
    return state + "-" + county + "-" + site

In [78]:
# conv ozone from ppm to ppb mostly 
def convert_aqs_units(df, species):
    df = df.copy()
    if "Units of Measure" not in df.columns:
        return df
    if species in ["NO2", "O3", "CO", "SO2"]:
        ppm_mask = df["Units of Measure"].str.contains(
            "Parts per million", case=False, na=False,)
        df.loc[ppm_mask, "Sample Measurement"] *= 1000
    return df

In [79]:
def calc_metrics(df):

    valid = df.dropna(subset=["OBS", "CMAQ"]).copy()
    if len(valid) == 0:
        return pd.Series({
            "N": 0,
            "Mean_OBS": np.nan,
            "Mean_CMAQ": np.nan,
            "MB": np.nan,
            "RMSE": np.nan,
            "R": np.nan,
            "NMB_%": np.nan,
            "NME_%": np.nan,})

    obs = valid["OBS"].astype(float)
    mod = valid["CMAQ"].astype(float)

    difference = mod - obs
    observation_sum = obs.sum()

    if observation_sum == 0:
        nmb = np.nan
        nme = np.nan
    else:
        nmb = (100.0 * difference.sum()/ observation_sum)
        nme = (100.0 * np.abs(difference).sum() / observation_sum)

    return pd.Series({
        "N": len(valid),
        "Mean_OBS": obs.mean(),
        "Mean_CMAQ": mod.mean(),
        "MB": difference.mean(),
        "RMSE": np.sqrt((difference ** 2).mean()),
        "R": obs.corr(mod),
        "NMB_%": nmb,
        "NME_%": nme,
    })

In [80]:
def prepare_aqs_data(
    aqs_file,
    species,
    start_dt,
    end_dt,):

    aqs = pd.read_csv(aqs_file)

    # Handle AQS files with underscore-formatted headers
    if "Date_GMT" in aqs.columns:

        aqs.columns = [
            column.replace("_", " ")
            for column in aqs.columns]

    # Create complete AQS site ID
    aqs["Site_ID"] = make_site_id(aqs)

    # Retain only the selected near-road sites
    aqs = aqs[
        aqs["Site_ID"].isin(site_names)].copy()

    # Create datetime
    aqs["Datetime GMT"] = pd.to_datetime(
        aqs["Date GMT"].astype(str)
        + " "
        + aqs["Time GMT"].astype(str),
        errors="coerce",)

    # Retain the requested month
    aqs = aqs[
        (aqs["Datetime GMT"] >= start_dt)
        & (aqs["Datetime GMT"] <= end_dt)].copy()

    # Convert ppm to ppb where necessary
    aqs = convert_aqs_units( aqs, species,)
    aqs["Species"] = species
    aqs["Site_Name"] = aqs["Site_ID"].map(site_names)

    return aqs

In [81]:
def match_aqs_and_cmaq(
    aqs,
    model,
    species,
    start_dt,
    end_dt,):

    matched_sites = []
    hourly_index = pd.date_range(
        start=start_dt,
        end=end_dt,
        freq="1h",)

    # Unique stations available for this species and month
    stations = (
        aqs[[
                "Site_ID",
                "Latitude",
                "Longitude",
            ]]
        .drop_duplicates()
        .reset_index(drop=True))

    print(f"{species}: {len(stations)} CMAP stations")

    for _, station in stations.iterrows():

        site_id = station["Site_ID"]
        station_lat = station["Latitude"]
        station_lon = station["Longitude"]
        site_data = aqs[aqs["Site_ID"] == site_id].copy()

        # ----------------------------------------------------
        # Find nearest CMAQ grid cell using Stacy's original code 
        # ----------------------------------------------------

        row, col = find_index(
            station_lon=station_lon,
            station_lat=station_lat,
            model_lon=lon_grid,
            model_lat=lat_grid,)

        # ----------------------------------------------------
        # Average duplicate observations within each hour
        # ----------------------------------------------------

        hourly_observations = (
            site_data
            .set_index("Datetime GMT")[
                "Sample Measurement"]
            .resample("1h")
            .mean()
            .reindex(hourly_index))

        # ----------------------------------------------------
        # Extract surface-layer CMAQ values
        # ----------------------------------------------------

        model_values = (
            model[species]
            .isel(
                LAY=0,
                ROW=row,
                COL=col,)
            .values
            .squeeze())

        if len(model_values) != len(hourly_index):
            raise ValueError(
                f"{species}, {site_id}: "
                f"model has {len(model_values)} hours, "
                f"but {len(hourly_index)} were expected.")

        # ----------------------------------------------------
        # Combine observation and model values
        # ----------------------------------------------------

        site_matched = pd.DataFrame({
            "Datetime GMT": hourly_index,
            "OBS": hourly_observations.to_numpy(),
            "CMAQ": model_values,})

        site_matched["Species"] = species
        site_matched["Site_ID"] = site_id
        site_matched["Latitude"] = station_lat
        site_matched["Longitude"] = station_lon
        site_matched["ROW"] = row
        site_matched["COL"] = col

        # Preserve useful AQS metadata
        for column in ["State Name", "County Name", "Local Site Name",]:

            if column in site_data.columns:
                valid_values = (site_data[column].dropna())

                if len(valid_values) > 0:
                    site_matched[column] = valid_values.iloc[0]
                else:
                    site_matched[column] = np.nan

        matched_sites.append(site_matched)

    if len(matched_sites) == 0:
        return pd.DataFrame()

    return pd.concat(
        matched_sites,
        ignore_index=True,)

In [82]:
def prepare_aqs_data(
    aqs_file,
    species,
    start_dt,
    end_dt,):

    aqs = pd.read_csv(aqs_file)

    # Handle files with underscore-formatted headers
    if "Date_GMT" in aqs.columns:

        aqs.columns = [
            column.replace("_", " ")
            for column in aqs.columns]

    # --------------------------------------------------------
    # Build standard AQS site ID
    # --------------------------------------------------------

    aqs["Site_ID"] = make_site_id(aqs)

    # --------------------------------------------------------
    # Create station points
    # --------------------------------------------------------

    aqs_points = gpd.GeoDataFrame(
        aqs.copy(),
        geometry=gpd.points_from_xy(
            aqs["Longitude"],
            aqs["Latitude"],),
        crs="EPSG:4326", )

    # --------------------------------------------------------
    # Retain only observations inside the CMAP polygon
    # --------------------------------------------------------

    aqs_points["in_cmap"] = (aqs_points.geometry.within(cmap_union))

    aqs_points = aqs_points[aqs_points["in_cmap"]].copy()

    # Return to a regular DataFrame
    aqs = pd.DataFrame(aqs_points.drop(columns="geometry"))

    # --------------------------------------------------------
    # Create datetime
    # --------------------------------------------------------

    aqs["Datetime GMT"] = pd.to_datetime(
        aqs["Date GMT"].astype(str)
        + " "
        + aqs["Time GMT"].astype(str),
        errors="coerce",)

    # --------------------------------------------------------
    # Retain requested month
    # --------------------------------------------------------
    aqs = aqs[
        (aqs["Datetime GMT"] >= start_dt)
        & (aqs["Datetime GMT"] <= end_dt)].copy()

    # Convert ppm to ppb where necessary
    aqs = convert_aqs_units(aqs,species,)
    aqs["Species"] = species

    return aqs

In [83]:
# ============================================================
# Process all four months and both scenarios
# ============================================================

all_hourly_data = []

for month, scenario_files in model_files.items():
    start_dt = pd.Timestamp(dates[month][0])
    end_dt = pd.Timestamp(dates[month][1])

    print()
    print("=" * 60)
    print("Processing month:", month)
    print("=" * 60)

    # --------------------------------------------------------
    # Read the observations once for this month
    # --------------------------------------------------------

    monthly_aqs = {}
    for species, aqs_file in aqs_files.items():
        print("Preparing AQS:", species)
        monthly_aqs[species] = prepare_aqs_data(
            aqs_file=aqs_file,
            species=species,
            start_dt=start_dt,
            end_dt=end_dt, )

    # --------------------------------------------------------
    # Process BASE and LOCUS
    # --------------------------------------------------------

    for scenario, model_file in scenario_files.items():

        print()
        print("Scenario:", scenario)
        print("Model file:", model_file)

        with xr.open_dataset(model_file) as model:

            for species in aqs_files:

                print("Matching:", species)

                # Skip species that are absent from the file
                if species not in model.data_vars:

                    print(
                        f"{species} is not in {model_file.name}. "
                        "Skipping." )
                    continue

                matched = match_aqs_and_cmaq(
                    aqs=monthly_aqs[species],
                    model=model,
                    species=species,
                    start_dt=start_dt,
                    end_dt=end_dt,)

                if matched.empty:
                    continue

                matched["Month"] = month
                matched["Scenario"] = scenario

                all_hourly_data.append(matched)


Processing month: 202301
Preparing AQS: NO2


C:\Users\x12la\AppData\Local\Temp\ipykernel_5632\292167600.py:11: DtypeWarning: Columns (17) have mixed types. Specify dtype option on import or set low_memory=False.
  aqs = pd.read_csv(aqs_file)


Preparing AQS: PM25_TOT


C:\Users\x12la\AppData\Local\Temp\ipykernel_5632\292167600.py:11: DtypeWarning: Columns (17) have mixed types. Specify dtype option on import or set low_memory=False.
  aqs = pd.read_csv(aqs_file)



Scenario: BASE
Model file: all_202301_base_reduced.nc
Matching: NO2
NO2: 5 CMAP stations
Matching: PM25_TOT
PM25_TOT: 8 CMAP stations

Scenario: LOCUS
Model file: all_202301_locus_reduced.nc
Matching: NO2
NO2: 5 CMAP stations
Matching: PM25_TOT
PM25_TOT: 8 CMAP stations

Processing month: 202304
Preparing AQS: NO2


...


Scenario: BASE
Model file: all_202307_base_reduced.nc
Matching: NO2
NO2: 6 CMAP stations
Matching: PM25_TOT
PM25_TOT: 8 CMAP stations

Scenario: LOCUS
Model file: all_202307_locus_reduced.nc
Matching: NO2
NO2: 6 CMAP stations
Matching: PM25_TOT
PM25_TOT: 8 CMAP stations

Processing month: 202310
Preparing AQS: NO2


C:\Users\x12la\AppData\Local\Temp\ipykernel_5632\292167600.py:11: DtypeWarning: Columns (17) have mixed types. Specify dtype option on import or set low_memory=False.
  aqs = pd.read_csv(aqs_file)


Preparing AQS: PM25_TOT


C:\Users\x12la\AppData\Local\Temp\ipykernel_5632\292167600.py:11: DtypeWarning: Columns (17) have mixed types. Specify dtype option on import or set low_memory=False.
  aqs = pd.read_csv(aqs_file)



Scenario: BASE
Model file: all_202310_base_reduced.nc
Matching: NO2
NO2: 6 CMAP stations
Matching: PM25_TOT
PM25_TOT: 8 CMAP stations

Scenario: LOCUS
Model file: all_202310_locus_reduced.nc
Matching: NO2
NO2: 6 CMAP stations
Matching: PM25_TOT
PM25_TOT: 8 CMAP stations


In [48]:
# ============================================================
# Combine all matched hourly data
# ============================================================
hourly_data = pd.concat( all_hourly_data, ignore_index=True,)
hourly_data["Datetime GMT"] = pd.to_datetime(hourly_data["Datetime GMT"])

print(hourly_data.shape)
display(hourly_data.head())

hourly_data.to_csv(
    output_dir / "CMAP_hourly_matched_data.csv",
    index=False,)

(79728, 13)


,Datetime GMT,OBS,CMAQ,Species,Site_ID,Latitude,Longitude,ROW,COL,State Name,County Name,Month,Scenario
0,2023-01-01 00:00:00,NaN,41.693935,NO2,17-031-0076,41.7514,-87.713488,108,139,Illinois,Cook,202301,BASE
1,2023-01-01 01:00:00,NaN,38.533123,NO2,17-031-0076,41.7514,-87.713488,108,139,Illinois,Cook,202301,BASE
2,2023-01-01 02:00:00,NaN,30.546045,NO2,17-031-0076,41.7514,-87.713488,108,139,Illinois,Cook,202301,BASE
3,2023-01-01 03:00:00,NaN,17.682198,NO2,17-031-0076,41.7514,-87.713488,108,139,Illinois,Cook,202301,BASE
4,2023-01-01 04:00:00,NaN,12.056955,NO2,17-031-0076,41.7514,-87.713488,108,139,Illinois,Cook,202301,BASE


In [46]:
# ============================================================
# Monthly statistics by CMAP station
# ============================================================

monthly_stats_by_site = (
    hourly_data
    .groupby(
        [
            "Month",
            "Scenario",
            "Species",
            "Site_ID",
            "Latitude",
            "Longitude",],
        sort=False,)
    .apply(calc_metrics)
    .reset_index())

monthly_stats_by_site.to_csv(output_dir / "CMAP_monthly_metrics_by_site.csv",index=False,)
display(monthly_stats_by_site)

,Month,Scenario,Species,Site_ID,Latitude,Longitude,N,Mean_OBS,Mean_CMAQ,MB,RMSE,R,NMB_%,NME_%
0,202301,BASE,NO2,17-031-0076,41.751400,-87.713488,672.0,14.109524,13.133428,-0.976096,8.754916,0.422776,-6.917995,47.703658
1,202301,BASE,NO2,17-031-0119,41.578620,-87.557406,738.0,15.373035,10.837676,-4.535360,10.283514,0.193424,-29.502044,55.346666
2,202301,BASE,NO2,17-031-0219,41.920009,-87.672995,738.0,16.296341,18.767002,2.470660,9.059015,0.514074,15.160827,39.702844
3,202301,BASE,NO2,17-031-3103,41.965193,-87.876265,738.0,16.653794,15.078957,-1.574837,8.390197,0.609448,-9.456326,38.742473
4,202301,BASE,NO2,17-031-4002,41.855243,-87.752470,735.0,16.142585,14.621137,-1.521448,7.516615,0.646918,-9.425055,34.779093
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
103,202310,LOCUS,PM25_TOT,17-031-4201,42.139996,-87.799227,742.0,7.822776,7.896748,0.073972,5.941612,0.502436,0.945592,51.871628
104,202310,LOCUS,PM25_TOT,17-043-4002,41.771071,-88.152534,742.0,7.853235,8.351343,0.498108,5.846133,0.454220,6.342717,51.664172
105,202310,LOCUS,PM25_TOT,17-111-0001,42.221442,-88.242207,742.0,6.859906,8.255447,1.395541,5.382201,0.596711,20.343444,53.378077
106,202310,LOCUS,PM25_TOT,17-197-1002,41.526885,-88.116474,740.0,8.024054,8.848634,0.824580,5.814860,0.495679,10.276349,50.429144


In [47]:
# ============================================================
# Four-month statistics by CMAP station
# ============================================================
annualized_stats_by_site = (
    hourly_data
    .groupby(
        [
            "Scenario",
            "Species",
            "Site_ID",
            "Latitude",
            "Longitude",
        ],
        sort=False,)
    .apply(calc_metrics)
    .reset_index())

annualized_stats_by_site["Period"] = ("Jan-Apr-Jul-Oct 2023")
annualized_stats_by_site.to_csv(output_dir / "CMAP_annualized_metrics_by_site.csv",index=False,)

display(annualized_stats_by_site)

,Scenario,Species,Site_ID,Latitude,Longitude,N,Mean_OBS,Mean_CMAQ,MB,RMSE,R,NMB_%,NME_%,Period
0,BASE,NO2,17-031-0076,41.751400,-87.713488,2354.0,12.515506,10.759372,-1.756134,7.550913,0.620471,-14.031664,43.270562,Jan-Apr-Jul-Oct 2023
1,BASE,NO2,17-031-0119,41.578620,-87.557406,2940.0,17.388027,9.014254,-8.373773,12.954850,0.341226,-48.158272,59.889425,Jan-Apr-Jul-Oct 2023
2,BASE,NO2,17-031-0219,41.920009,-87.672995,2678.0,16.644772,17.038966,0.394193,10.194119,0.600024,2.368272,43.615347,Jan-Apr-Jul-Oct 2023
3,BASE,NO2,17-031-3103,41.965193,-87.876265,2920.0,17.340000,12.739784,-4.600216,9.566763,0.626730,-26.529504,42.556440,Jan-Apr-Jul-Oct 2023
4,BASE,NO2,17-031-4002,41.855243,-87.752470,2018.0,14.913380,12.257175,-2.656204,8.000819,0.700974,-17.810882,38.252377,Jan-Apr-Jul-Oct 2023
5,BASE,PM25_TOT,17-031-0119,41.578620,-87.557406,2945.0,11.188862,11.035580,-0.153282,8.106634,0.586161,-1.369954,46.070589,Jan-Apr-Jul-Oct 2023
6,BASE,PM25_TOT,17-031-3103,41.965193,-87.876265,2855.0,10.302434,10.714127,0.411693,7.634323,0.654029,3.996076,46.164390,Jan-Apr-Jul-Oct 2023
7,BASE,PM25_TOT,17-031-4007,42.060285,-87.863225,2849.0,10.044665,9.843954,-0.200711,7.833178,0.614246,-1.998186,47.203340,Jan-Apr-Jul-Oct 2023
8,BASE,PM25_TOT,17-031-4201,42.139996,-87.799227,2937.0,9.652911,9.413999,-0.238912,7.489805,0.606350,-2.475029,49.212699,Jan-Apr-Jul-Oct 2023
9,BASE,PM25_TOT,17-043-4002,41.771071,-88.152534,2628.0,9.261130,9.626556,0.365426,6.019320,0.521520,3.945800,45.140244,Jan-Apr-Jul-Oct 2023
